In [1]:
import numpy as np

In [2]:
def standard_softmax(x):
    """표준 방식의 안정적인 소프트맥스 계산"""
    m = np.max(x)
    x_exp = np.exp(x - m)
    return x_exp / np.sum(x_exp)

In [3]:
def online_softmax(x, block_size):
    """
    온라인(스트리밍) 방식으로 소프트맥스를 계산합니다.

    Args:
        x (np.ndarray): 입력 벡터.
        block_size (int): 한 번에 처리할 블록(타일)의 크기.

    Returns:
        np.ndarray: 온라인 방식으로 계산된 소프트맥스 결과.
    """
    # 1. 통계치 및 결과 변수 초기화
    m = -np.inf  # 현재까지의 최댓값 (running max)
    l = 0.0      # 정규화된 값들의 지수 합 (sum of exponentials)

    # 최종 소프트맥스 결과가 저장될 배열
    # 실제로는 이 배열에 P_ij * V_j 가 누적됩니다.
    # 여기서는 소프트맥스 값 자체를 저장하여 원리를 보여줍니다.
    p_online = np.zeros_like(x, dtype=np.float32)

    print(f"입력 벡터 크기: {len(x)}, 블록 크기: {block_size}")
    print("-" * 30)

    # 2. 블록 단위로 순회
    for i in range(0, len(x), block_size):
        print(f"처리 중인 블록: 인덱스 {i}부터 {i+block_size-1}까지")

        # 현재 처리할 블록(타일)을 정의
        x_block = x[i:i + block_size]

        # 3. 현재 블록의 지역 최댓값 계산
        m_block = np.max(x_block)

        # 4. 이전까지의 전역 최댓값과 비교하여 새로운 전역 최댓값 갱신
        m_old = m
        m_new = np.maximum(m_old, m_block)

        # 5. 최댓값이 갱신되었다면, 이전까지의 지수 합 'l'을 재조정
        # 이전 결과를 저장했던 p_online도 함께 재조정합니다.
        rescale_factor = np.exp(m_old - m_new)
        l = l * rescale_factor
        p_online[:i] *= rescale_factor # 이전 블록까지의 결과 재조정

        # 6. 현재 블록을 새로운 최댓값 m_new로 정규화하고 지수 함수 적용
        p_block = np.exp(x_block - m_new)

        # 7. 현재 블록의 지수 합을 전체 지수 합 'l'에 누적
        l += np.sum(p_block)

        # p_online의 현재 블록 위치에 계산된 값을 저장
        p_online[i:i + block_size] = p_block

        m = m_new # 전역 최댓값 갱신

        print(f"  블록 최댓값: {m_block:.4f}, 이전 전역 최댓값: {m_old:.4f}, 새 전역 최댓값: {m_new:.4f}")
        print(f"  재조정 계수: {rescale_factor:.4f}")
        print(f"  갱신된 지수 합 l: {l:.4f}")
        print("-" * 30)

    # 8. 모든 블록 처리 후, 최종 지수 합 'l'로 나누어 정규화
    final_softmax = p_online / l

    return final_softmax


In [4]:
# --- 실행 및 결과 비교 ---
# 무작위 입력 데이터 생성
N = 16  # 전체 시퀀스 길이
block_size = 4 # SRAM에 들어갈 수 있다고 가정한 타일 크기
x = np.random.randn(N).astype(np.float32)

print("입력 벡터 x:\n", x)
print("\n" + "="*50 + "\n")

# 온라인 소프트맥스 계산
p_online_result = online_softmax(x, block_size)

# 표준 소프트맥스 계산
p_standard_result = standard_softmax(x)

print("온라인 소프트맥스 계산 결과:\n", p_online_result)
print("\n표준 소프트맥스 계산 결과:\n", p_standard_result)

# 두 결과가 수치적으로 매우 가까운지 확인
is_close = np.allclose(p_online_result, p_standard_result)
print(f"\n결과 일치 여부: {is_close}")
if is_close:
    print("성공: 온라인 소프트맥스가 표준 소프트맥스와 동일한 결과를 정확하게 계산했습니다.")

입력 벡터 x:
 [ 1.4683435e+00 -7.5477290e-01  1.3980856e+00  5.3311068e-01
  5.7597160e-02 -1.0505029e+00 -1.0061278e-01  1.3444284e+00
  5.0024194e-01  5.0416946e-01  6.4516667e-04  8.0738020e-01
  1.1112740e+00  4.4346583e-01  4.7793254e-01 -8.9450628e-01]


입력 벡터 크기: 16, 블록 크기: 4
------------------------------
처리 중인 블록: 인덱스 0부터 3까지
  블록 최댓값: 1.4683, 이전 전역 최댓값: -inf, 새 전역 최댓값: 1.4683
  재조정 계수: 0.0000
  갱신된 지수 합 l: 2.4329
------------------------------
처리 중인 블록: 인덱스 4부터 7까지
  블록 최댓값: 1.3444, 이전 전역 최댓값: 1.4683, 새 전역 최댓값: 1.4683
  재조정 계수: 1.0000
  갱신된 지수 합 l: 3.8491
------------------------------
처리 중인 블록: 인덱스 8부터 11까지
  블록 최댓값: 0.8074, 이전 전역 최댓값: 1.4683, 새 전역 최댓값: 1.4683
  재조정 계수: 1.0000
  갱신된 지수 합 l: 5.3571
------------------------------
처리 중인 블록: 인덱스 12부터 15까지
  블록 최댓값: 1.1113, 이전 전역 최댓값: 1.4683, 새 전역 최댓값: 1.4683
  재조정 계수: 1.0000
  갱신된 지수 합 l: 6.8812
------------------------------
온라인 소프트맥스 계산 결과:
 [0.1453235  0.01573434 0.13546379 0.05703867 0.03545328 0.01170617
 0.03026542 0.12838674 